# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rupikaupa2006/Flyrank-ML-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [9]:
%pip -q install duckdb huggingface_hub pandas

In [10]:
import os, getpass

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

HF_TOKEN = HF_TOKEN or getpass.getpass(
    "Paste your Hugging Face READ token (hf_...): "
)

In [11]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_daily_sample": f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

print("✅ Connected successfully!")

✅ Connected successfully!


## Baseline Rule

I will prioritize content that has low click-through rate (CTR), low search position performance, and older content age. These pages are more likely to benefit from a content refresh because they have search demand but may not be attracting enough clicks or may contain outdated information.

### Reason Codes

- REFRESH_OLD_CONTENT — Content is old and likely needs updating.
- LOW_CTR — Click-through rate is below the expected level.
- LOW_VISIBILITY — Search performance is weak despite existing impressions.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

## Baseline Scoring Strategy

I assign a higher priority score to content that has high search volume, has not been optimized recently, and is eligible for optimization. The score supports a content refresh queue by ranking pages that are more likely to benefit from review.

The output includes:
- A numeric priority score.
- A reason code explaining the recommendation.
- An action label indicating the suggested action.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

baseline = con.sql(f"""
SELECT
    content_hash_id,
    search_volume,
    competition_level,
    word_count,
    content_created_date,
    optimization_eligible_date,
    is_published,

    CASE
        WHEN optimization_eligible_date IS NOT NULL THEN 50
        ELSE 0
    END
    +
    CASE
        WHEN search_volume > 1000 THEN 30
        ELSE 10
    END
    +
    CASE
        WHEN word_count < 1500 THEN 20
        ELSE 5
    END AS baseline_score,

    CASE
        WHEN optimization_eligible_date IS NOT NULL THEN 'REFRESH_OLD_CONTENT'
        WHEN search_volume > 1000 THEN 'HIGH_SEARCH_VOLUME'
        ELSE 'LOW_CONTENT_DEPTH'
    END AS reason_code,

    'Review and Refresh' AS action

FROM {TABLES['dim_content']}
WHERE is_published = TRUE
ORDER BY baseline_score DESC
LIMIT 100
""").df()

baseline.head()
import os

os.makedirs("work/outputs", exist_ok=True)

baseline.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("✅ baseline_action_score.csv saved successfully!")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ baseline_action_score.csv saved successfully!


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*



The highest-ranked content items were prioritized because they received high baseline scores based on optimization eligibility, search volume, and content depth.

Each recommendation should be reviewed manually before implementation. A recommendation may be incorrect if:
- the content was recently updated outside the recorded data,
- business priorities have changed,
- search demand has shifted,
- or the page already performs well for its intended audience.

The baseline score is intended as decision-support rather than an automatic action.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = baseline.head(20).copy()

top20["confidence_note"] = "Medium"
top20["what_would_make_it_wrong"] = (
    "Recent manual optimization, changing search trends, or missing contextual business information."
)

top20[
    [
        "content_hash_id",
        "action",
        "reason_code",
        "confidence_note",
        "what_would_make_it_wrong",
    ]
]


,content_hash_id,action,reason_code,confidence_note,what_would_make_it_wrong
0,content_d65ca6a7e33f6659,Review and Refresh,REFRESH_OLD_CONTENT,Medium,"Recent manual optimization, changing search tr..."
1,content_b4672b8af97a3e96,Review and Refresh,REFRESH_OLD_CONTENT,Medium,"Recent manual optimization, changing search tr..."
2,content_8e74c3b078468eea,Review and Refresh,REFRESH_OLD_CONTENT,Medium,"Recent manual optimization, changing search tr..."
3,content_161c2683680cac2d,Review and Refresh,REFRESH_OLD_CONTENT,Medium,"Recent manual optimization, changing search tr..."
4,content_9aa9607932c17d41,Review and Refresh,REFRESH_OLD_CONTENT,Medium,"Recent manual optimization, changing search tr..."
5,content_471d9cabce329a66,Review and Refresh,REFRESH_OLD_CONTENT,Medium,"Recent manual optimization, changing search tr..."
6,content_18056e1213bc937a,Review and Refresh,REFRESH_OLD_CONTENT,Medium,"Recent manual optimization, changing search tr..."
7,content_e73024da2a848e26,Review and Refresh,REFRESH_OLD_CONTENT,Medium,"Recent manual optimization, changing search tr..."
8,content_293360e95b00a950,Review and Refresh,REFRESH_OLD_CONTENT,Medium,"Recent manual optimization, changing search tr..."
9,content_ada869d4083926a2,Review and Refresh,REFRESH_OLD_CONTENT,Medium,"Recent manual optimization, changing search tr..."


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*



Some recommendations may be weak because the baseline rule is intentionally simple. High search volume or optimization eligibility alone does not guarantee that a page needs refreshing. Pages may already perform well, have different business priorities, or have been updated outside the available data.

I confirmed that the baseline rule does not use future information or product-generated flags. The score is based only on features that are available before making the recommendation, making it suitable as a baseline decision-support system.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Leakage Check")
print("-" * 40)
print("✓ No future-window features used")
print("✓ No label-derived columns used")
print("✓ No product flags used")
print("✓ Baseline score uses only observable content features")

Leakage Check
----------------------------------------
✓ No future-window features used
✓ No label-derived columns used
✓ No product flags used
✓ Baseline score uses only observable content features


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.